# 🔄 RAG Self-Correction Loop — Isolated Test Notebook

**Author:** Umang Vijay | JECRC University  
**Project:** Autonomous Data Science Co-Pilot  

This notebook isolates and tests the RAG (Retrieval-Augmented Generation) self-correction pipeline:
1. Build FAISS index from documentation corpus
2. Test retrieval with sample error queries
3. Test code generation via Gemini
4. Test sandbox execution
5. Test the full self-correction loop
6. Measure success metrics

## Cell 1: Setup & Dependencies

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install langchain langchain-google-genai langchain-community langchain-text-splitters faiss-cpu google-generativeai pandas numpy matplotlib plotly python-dotenv scikit-learn statsmodels

import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load API key
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('GOOGLE_API_KEY', '')

if not API_KEY or API_KEY == 'your_gemini_api_key_here':
    API_KEY = input('Enter your Google Gemini API Key: ')

print(f'✅ Project Root: {PROJECT_ROOT}')
print(f'✅ API Key loaded: {"Yes" if API_KEY else "No"}')

## Cell 2: Build FAISS Index from Documentation Corpus

In [ ]:
from core.rag_pipeline import RAGPipeline

# Initialize RAG pipeline
rag = RAGPipeline(
    api_key=API_KEY,
    docs_dir=str(PROJECT_ROOT / 'docs_corpus'),
    index_dir=str(PROJECT_ROOT / 'faiss_index')
)

# Build or load the index
success = rag.build_or_load_index()
print(f'\n📊 Index Stats: {rag.get_index_stats()}')
print(f'✅ RAG Pipeline Ready: {success}')

## Cell 3: Test Retrieval — Query with Sample Errors

In [ ]:
# Test retrieval with common error patterns
test_errors = [
    ("KeyError: 'Revenue'", "df.groupby('Region')['Revenue'].sum()"),
    ("TypeError: unsupported operand type(s) for +: 'int' and 'str'", "total = df['Amount'] + df['Tax']"),
    ("AttributeError: Can only use .dt accessor with datetimelike values", "df['Date'].dt.month"),
    ("ValueError: Input contains NaN", "model.fit(X, y)"),
]

for error_msg, code in test_errors:
    print(f'\n{"="*60}')
    print(f'🔴 Error: {error_msg}')
    print(f'📝 Code: {code}')
    print(f'\n📚 RAG Retrieved Context:')
    context = rag.query_for_fix(error_msg, code, top_k=2)
    print(context[:500])
    print(f'\n✅ Context retrieved successfully')

## Cell 4: Test Code Generation via Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    google_api_key=API_KEY,
    temperature=0.2
)

# Test code generation
system_msg = SystemMessage(content="""You are a Python data analyst. Write executable Python code only.
The DataFrame is pre-loaded as `df`. Use save_chart(fig, 'name') for matplotlib.
Use report_insights({'key': 'value'}) for insights. CODE ONLY.""")

test_prompt = """Generate code to show revenue by region as a bar chart.
Columns available: Date, Region, Product, Revenue, Units_Sold"""

response = llm.invoke([system_msg, HumanMessage(content=test_prompt)])
generated_code = response.content

print('📝 Generated Code:')
print(generated_code)

## Cell 5: Test Sandbox Execution

In [ ]:
import pandas as pd
from core.sandbox import SandboxExecutor

# Load sample data
sample_df = pd.read_csv(PROJECT_ROOT / 'sample_data' / 'sales_data.csv')
print(f'📊 Loaded sample data: {sample_df.shape}')

# Initialize sandbox
sandbox = SandboxExecutor(timeout=30)

# Save df for sandbox access
df_path = sandbox.save_dataframe(sample_df, 'test_run')
print(f'💾 Data saved to: {df_path}')

# Simple test code
test_code = """
print(f'DataFrame shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
revenue_by_region = df.groupby('Region')['Revenue'].sum().sort_values(ascending=False)
print(f'Revenue by Region:')
print(revenue_by_region)

fig, ax = plt.subplots(figsize=(10, 6))
revenue_by_region.plot(kind='bar', ax=ax, color=['#6C63FF', '#FF6584', '#43E97B', '#FFD93D', '#6EC6FF'])
ax.set_title('Revenue by Region', fontsize=16, fontweight='bold')
ax.set_ylabel('Total Revenue ($)')
plt.xticks(rotation=45)
save_chart(fig, 'revenue_by_region')

report_insights({'total_revenue': f'${revenue_by_region.sum():,.2f}',
                 'top_region': revenue_by_region.index[0],
                 'top_revenue': f'${revenue_by_region.iloc[0]:,.2f}'})
"""

result = sandbox.execute(test_code, df_path)
print(f'\n✅ Success: {result.success}')
print(f'⏱️ Execution Time: {result.execution_time}s')
print(f'📊 Charts: {result.chart_paths}')
print(f'\n📤 Stdout:\n{result.stdout}')
if result.stderr:
    print(f'\n❌ Stderr:\n{result.stderr}')

## Cell 6: Test Self-Correction Loop — Intentional Error

In [ ]:
# Intentionally broken code (wrong column name + deprecated API)
broken_code = """
# This code has intentional errors for testing self-correction
revenue = df.groupby('Regon')['Revnue'].sum()  # Typos in column names
df['month'] = df['Date'].dt.month  # Date not parsed as datetime
monthly = df.groupby('month').sum()  # Missing numeric_only (pandas 2.0+)
print(revenue)
"""

# Execute broken code
result = sandbox.execute(broken_code, df_path)
print(f'Expected failure: Success = {result.success}')
print(f'\n❌ Error:\n{result.stderr[:500]}')

# RAG query for fix
print(f'\n{"="*60}')
print('🔄 Querying RAG for fix...')
fix_context = rag.query_for_fix(result.stderr, broken_code)
print(f'\n📚 RAG Context (first 500 chars):\n{fix_context[:500]}')

# Use LLM to generate fixed code
fix_prompt = f"""Fix this broken Python code:

```python
{broken_code}
```

Error: {result.stderr[:300]}

Documentation context:
{fix_context[:1000]}

Available columns: {sample_df.columns.tolist()}
Write the COMPLETE fixed code. CODE ONLY."""

response = llm.invoke([system_msg, HumanMessage(content=fix_prompt)])
fixed_code = response.content

# Clean markdown fences
import re
match = re.search(r'```python\s*\n(.*?)```', fixed_code, re.DOTALL)
if match:
    fixed_code = match.group(1)

print(f'\n📝 Fixed Code:\n{fixed_code}')

# Execute fixed code
result2 = sandbox.execute(fixed_code, df_path)
print(f'\n✅ Fixed code success: {result2.success}')
print(f'📤 Output:\n{result2.stdout}')
if result2.stderr:
    print(f'⚠️ Warnings:\n{result2.stderr[:300]}')

## Cell 7: End-to-End Test — Full Agent Pipeline

In [ ]:
from core.agent import DataScienceCoPilot

# Initialize the full agent
agent = DataScienceCoPilot(api_key=API_KEY, max_retries=3)
agent.initialize_rag()

# Test queries
test_queries = [
    ('Show revenue by region as a bar chart', 'sales_dashboard'),
    ('Analyze data quality and find missing values', 'data_quality'),
    ('What are the top 3 products by total units sold?', 'ad_hoc'),
]

for query, use_case in test_queries:
    print(f'\n{"="*60}')
    print(f'📝 Query: {query}')
    print(f'🎯 Use Case: {use_case}')

    result = agent.analyze(sample_df, query, use_case)

    print(f'✅ Success: {result.success}')
    print(f'🔄 Attempts: {result.total_attempts}')
    print(f'📊 Charts: {len(result.chart_paths)}')
    if result.insights:
        print(f'💡 Insights: {list(result.insights.keys())}')
    if not result.success:
        print(f'❌ Error: {result.error[:200]}')

# Cleanup
agent.cleanup()
print(f'\n🧹 Sandbox cleaned up')

## Cell 8: Metrics — Correction Success Rate

In [ ]:
import time

# Run a batch of tests and measure success rate
agent2 = DataScienceCoPilot(api_key=API_KEY, max_retries=3)
agent2.initialize_rag()

batch_queries = [
    ('Show total revenue per product', 'sales_dashboard'),
    ('Find all duplicate rows', 'data_quality'),
    ('Plot monthly revenue trend', 'trend_analysis'),
    ('What is the average customer rating by region?', 'ad_hoc'),
    ('Show the distribution of discount percentages', 'ad_hoc'),
]

results_log = []
for query, use_case in batch_queries:
    start = time.time()
    result = agent2.analyze(sample_df, query, use_case)
    elapsed = time.time() - start

    results_log.append({
        'query': query,
        'use_case': use_case,
        'success': result.success,
        'attempts': result.total_attempts,
        'time_s': round(elapsed, 2),
        'charts': len(result.chart_paths),
    })

# Summary
results_df = pd.DataFrame(results_log)
print('\n📊 Batch Test Results:')
print(results_df.to_string(index=False))
print(f'\n📈 Success Rate: {results_df["success"].mean()*100:.0f}%')
print(f'⏱️ Avg Time: {results_df["time_s"].mean():.1f}s')
print(f'🔄 Avg Attempts: {results_df["attempts"].mean():.1f}')

agent2.cleanup()